# demo_hb

> Phase 13: the hidden-profile vote's fixes on an established benchmark, HiddenBench's 65 tasks (Li, Naito, Shirado, ICML 2026; MIT), with Claude Haiku 5.5: `baseline`, `factsFirst`, `table`, `dissent` and the one-agent ceiling, each task played 3 times with shuffled options and facts (`HB_PLAN`, `specs/multiagent/phase13/hiddenbench.md`).

Artifacts (gitignored): `results/multiagent/hiddenbench/{quick,pilot,full}/`.

In [ ]:
import json
import os
import shutil
import subprocess
import time

import numpy as np

from explRL.bench_grid import readCSV, writeCSV
from explRL.coop_grid import coopHash
from explRL.fix_grid import HB_PLAN, hbGrid, hbPilotGrid
from explRL.fix_stats import gapClosed, holmFamily
from explRL.llm_budget import Budget
from explRL.llm_runs import LLMConfig, makeClient
from explRL.profile_runs import runProfile
from explRL.train import demo_artifact_dir

DN, CAP, WORKERS = 'multiagent/hiddenbench', 150.0, 16
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))
f = lambda x: float(x) if x not in ('', None) else float('nan')


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def hbSummary(rows, study):
    "Per arm: win rate, agent accuracy, disclosure; gap closed vs baseline against the solo ceiling (Holm over HB_PLAN's pair)."
    rs = [r for r in rows if r['study'] == study]
    by = lambda arm, k: {int(f(r['seed'])): f(r[k]) for r in rs if r['level'] == f'{arm}|hb'}
    acc = lambda arm: {s: v / 4 for s, v in by(arm, 'votesA').items()}
    arms = [a for a in HB_PLAN['arms'] if by(a, 'success')]
    S = dict(arms={a: dict(win=float(np.mean(list(by(a, 'success').values()))), n=len(by(a, 'success')),
                           accuracy=float(np.nanmean(list(acc(a).values()))) if a != 'solo' else float('nan'),
                           disclosed=float(np.nanmean(list(by(a, 'uniquePosted').values()))) if a != 'solo' else float('nan'),
                           invalid=float(np.nanmean([f(r['invalid']) / max(f(r['calls']), 1) for r in rs if r['level'] == f'{a}|hb'])))
                   for a in arms}, gap={})
    if 'baseline' in arms and 'solo' in arms:
        for a in ('factsFirst', 'table', 'dissent'):
            if a in arms:
                S['gap'][a] = gapClosed(by(a, 'success'), by('baseline', 'success'), by('solo', 'success'))
        holmFamily(S['gap'], [a for a in HB_PLAN['holm'] if a in S['gap']])
    return S


def writeHB(path, S, meta):
    g = lambda x: 'n/a' if x != x else f'{x:.2f}'
    L = [f"# HiddenBench replication of the vote fixes ({meta['mode']})", '', 'Exploratory, pre-specified in `HB_PLAN`.', '',
         f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`; Claude Haiku 5.5; HiddenBench (65 tasks, MIT, "
         f"Li, Naito, Shirado, ICML 2026); cost ${meta['cost']:.2f}; total API spend ${meta['spent']:.2f} of the $150 cap", '',
         '| arm | win rate (majority) | agent accuracy | hidden facts disclosed | invalid replies | episodes |', '|---|---|---|---|---|---|']
    L += [f"| {a} | {g(v['win'])} | {g(v['accuracy'])} | {g(v['disclosed'])} | {v['invalid']:.3f} | {v['n']} |" for a, v in S['arms'].items()]
    L += ['', '| arm | gap closed [95% CI] | n | p | Holm p |', '|---|---|---|---|---|']
    L += [f"| {a} | {g(v['gap'])} [{g(v['ci'][0])}, {g(v['ci'][1])}] | {v['n']} | {v['p']:.3f} | "
          f"{g(v['pHolm']) if v['primary'] else 'control'} |" for a, v in S['gap'].items()]
    path.write_text('\n'.join(L) + '\n')


def run(mode, cells, spec, workers=WORKERS, study='hbA'):
    "Run the cells through runProfile; runs.csv, turns.jsonl, report.md, summary.json."
    out = demo_artifact_dir(mode, DN)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    client = makeClient(LLMConfig.fromDict(cells[0]['cfg']), spec)
    t0 = time.time()
    rows = runProfile(cells, client, logs, workers=workers)
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as fh:
        for p in sorted(logs.glob('*.jsonl')):
            fh.write(p.read_text())
    shutil.rmtree(logs, ignore_errors=True)
    S = hbSummary(rows, study)
    meta = dict(mode=mode, planHash=coopHash(HB_PLAN), commit=gitCommit(), cost=float(sum(f(r['cost']) for r in rows)),
                spent=Budget(CAP, spec['ledger']).spent() if spec.get('ledger') else 0.0, wall=time.time() - t0)
    writeHB(out / 'report.md', S, meta)
    (out / 'summary.json').write_text(json.dumps(dict(S=S, meta=meta), indent=1, default=str) + '\n')
    return rows, S

## Quick: two tasks with the fake client (pipeline check)

In [ ]:
rows, S = run('quick', hbGrid(tasks=(1, 2), reps=2), dict(kind='fake'), study='hbA')
assert {r['level'] for r in rows} == {f'{a}|hb' for a in HB_PLAN['arms']} and len(rows) == len(HB_PLAN['arms']) * 4
again, _ = run('quick', hbGrid(tasks=(1, 2), reps=2), dict(kind='fake'), workers=1)
assert [r['success'] for r in again] == [r['success'] for r in rows]
print(open(demo_artifact_dir('quick', DN) / 'report.md').read())

## Pilot (`--flags llm`, live, 340 calls): baseline and solo on the first 20 tasks; check the bands before freezing `HB_PLAN`

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    rows, S = run('pilot', hbPilotGrid(), LIVE_SPEC, study='hbPilot')
    b, so = S['arms']['baseline'], S['arms']['solo']
    print(f"baseline win {b['win']:.2f}, agent accuracy {b['accuracy']:.2f}, invalid {b['invalid']:.3f}; solo {so['win']:.2f} "
          f"(bands: invalid < {HB_PLAN['bands']['invalid']}, solo >= {HB_PLAN['bands']['solo']}, gap >= {HB_PLAN['bands']['gap']}); "
          f"cost ${sum(f(r['cost']) for r in rows):.2f}")

## Main (`--cell-timeout 0 --flags llm`, live, about 12 700 calls)

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert HB_PLAN['frozen'], 'freeze and commit HB_PLAN before the main run'
    rows, S = run('full', hbGrid(), LIVE_SPEC)
    print(open(demo_artifact_dir('full', DN) / 'report.md').read())

## Cached rerun (`--flags llm`, after the main run)

In [ ]:
#| llm
full = demo_artifact_dir('full', DN)
if not LIVE or not (full / 'runs.csv').exists():
    print('skipped: needs EXPLRL_LLM_LIVE=1 and the main run')
else:
    before, spent = [(full / x).read_bytes() for x in ('runs.csv', 'turns.jsonl')], Budget(CAP, LIVE_SPEC['ledger']).spent()
    run('full', hbGrid(), LIVE_SPEC)
    assert [(full / x).read_bytes() for x in ('runs.csv', 'turns.jsonl')] == before and Budget(CAP, LIVE_SPEC['ledger']).spent() == spent
    print('cached rerun: byte-identical, free')